# Youth LFPR vs Unemployment Analysis

This notebook provides an in-depth analysis of the relationship between Youth LFPR and Unemployment in Sri Lanka, matching the comprehensive exploratory and statistical depth seen in other macro-labour combinations.

## 1. Data Loading & Initial Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use('default')

# Load main unemployment data (1991-2023)
unemp_df = pd.read_csv('../labour/csv/API_SL.UEM.TOTL.ZS_DS2_en_csv_v2_93.csv', skiprows=4)
unemp_sl = unemp_df[unemp_df['Country Name'] == 'Sri Lanka'].copy()
years = [str(y) for y in range(1991, 2024)]
unemp_sl = unemp_sl[['Country Name'] + years].melt(id_vars=['Country Name'], var_name='Year', value_name='Unemployment_Rate')
unemp_sl['Year'] = pd.to_datetime(unemp_sl['Year'], format='%Y')
unemp_sl.set_index('Year', inplace=True)
unemp_sl.drop(columns=['Country Name'], inplace=True)

print("Unemployment Data Info:")
print(unemp_sl.info())
unemp_sl.head()

In [ ]:
# Load Feature Data
feature_df = pd.read_csv('../labour/finalized_csv/Labor_force_participation_rate,_total_(% of total population ages 15-64)_(modeled ILO estimate)_sl_indicators/Labor force participation rate for ages 15-24, total (%) (modeled ILO estimate).csv')
feature_df = feature_df[['Year', 'Value']].copy()
feature_df.rename(columns={'Value': 'Youth_LFPR_%'}, inplace=True)
feature_df['Year'] = pd.to_datetime(feature_df['Year'].astype(str), format='%Y')
feature_df.set_index('Year', inplace=True)

print(f"Feature Data Info:")
print(feature_df.info())
feature_df.head()

## 2. Missing Values Analysis & Imputation

In [ ]:
print("Missing values in Unemployment:", unemp_sl['Unemployment_Rate'].isnull().sum())
print("Missing values in Youth_LFPR_%:", feature_df['Youth_LFPR_%'].isnull().sum())

# Applying Linear Interpolation (Best practice for macroeconomic time series)
unemp_clean = unemp_sl.interpolate(method='linear').bfill().ffill()
feature_clean = feature_df.interpolate(method='linear').bfill().ffill()

print("\nPost-imputation completeness verified.")

## 3. Data Alignment and Merging

In [ ]:
# Inner join to guarantee alignment
combined_df = unemp_clean.join(feature_clean, how='inner')

print(f"Combined Dataset: {len(combined_df)} records (from {combined_df.index.year.min()} to {combined_df.index.year.max()})")
combined_df.head()

## 4. Comprehensive Correlation Analysis

In [ ]:
u_vals = combined_df['Unemployment_Rate'].values
f_vals = combined_df['Youth_LFPR_%'].values

pearson_r, pearson_p = stats.pearsonr(f_vals, u_vals)
spearman_r, spearman_p = stats.spearmanr(f_vals, u_vals)

print("=" * 60)
print("CORRELATION STATISTICS")
print("=" * 60)
print(f"Pearson (Linear):       r = {pearson_r:.4f}  | p-value = {pearson_p:.6f}")
print(f"Spearman (Monotonic):   Rho = {spearman_r:.4f} | p-value = {spearman_p:.6f}")
print("=" * 60)


## 5. Extensive Visualization (Structural Patterns)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Scatter with Regression Line
sns.regplot(x=f_vals, y=u_vals, ax=axes[0, 0], scatter_kws={'alpha': 0.7, 's': 60}, line_kws={'color': 'red'}, color='#e74c3c')
axes[0, 0].set_xlabel('Youth_LFPR_%', fontweight='bold')
axes[0, 0].set_ylabel('Unemployment Rate (%)', fontweight='bold')
axes[0, 0].set_title('Cross-Sectional Scatter & Linear Fit', fontweight='bold')

# 2. KDE Density Plot (Bivariate)
sns.kdeplot(x=f_vals, y=u_vals, ax=axes[0, 1], cmap='Blues', fill=True, alpha=0.8)
axes[0, 1].set_xlabel('Youth_LFPR_%')
axes[0, 1].set_ylabel('Unemployment Rate (%)')
axes[0, 1].set_title('Bivariate Density Heatmap', fontweight='bold')

# 3. Time Series Dynamics
ax_time = axes[1, 0]
ax_time.plot(combined_df.index.year, combined_df['Unemployment_Rate'], color='tab:blue', marker='o', label='Unemployment')
ax_time.set_ylabel('Unemployment Rate (%)', color='tab:blue', fontweight='bold')
ax_time.tick_params(axis='y', labelcolor='tab:blue')
ax_twin = ax_time.twinx()
ax_twin.plot(combined_df.index.year, combined_df['Youth_LFPR_%'], color='#e74c3c', marker='s', linestyle='--', label='Youth_LFPR_%')
ax_twin.set_ylabel('Youth_LFPR_%', color='#e74c3c', fontweight='bold')
ax_twin.tick_params(axis='y', labelcolor='#e74c3c')
ax_time.set_title('Historical Overlap (Dual Axis)', fontweight='bold')

# 4. Boxplots for Data Distribution
bp_data = pd.DataFrame({'Unemployment': (u_vals - u_vals.mean()) / u_vals.std(), 
                      'Youth LFPR Context': (f_vals - f_vals.mean()) / f_vals.std()})
sns.boxplot(data=bp_data, ax=axes[1, 1], palette=['tab:blue', '{color}'])
axes[1, 1].set_title('Standardized Distribution Variance', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Advanced Models: Linear Regression & Residuals

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

X = f_vals.reshape(-1, 1)
y = u_vals

model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)
residuals = y - y_pred

print("REGRESSION DIAGNOSTICS")
print(f"R-squared Score: {r2_score(y, y_pred):.4f}")
print(f"RMSE:            {np.sqrt(mean_squared_error(y, y_pred)):.4f}")
print(f"Coefficient:     {model.coef_[0]:.4f}")
print(f"Intercept:       {model.intercept_:.4f}")


## 7. Lagged Analysis (Time Delays in Macro Effects)

In [ ]:
print("LAG ANALYSIS: Macro-factors often take time to impact unemployment.")
print("=" * 70)

for lag in range(0, 5):
    shifted_feature = combined_df['Youth_LFPR_%'].shift(lag)
    corr = combined_df['Unemployment_Rate'].corr(shifted_feature)
    print(f"Lag {lag} Years: correlation = {corr:.4f}")

print("\nThis identifies how many years it takes for changes in this sector to penetrate the labour market.")
